# Auditor Agent — GRPO Training
Trains the Auditor agent to classify invoices as `approved` or `flagged` with fraud type.
Uses the same live HF Space `/multi/audit` endpoint as the reward verifier.

**Model:** Qwen2.5-1.5B-Instruct, 4-bit QLoRA, r=16  
**Reward:** +0.99 correct fraud (right type), +0.90 clean clearance, +0.01 miss/FP  
**Environment:** https://ps2181-invoice-processing-pipeline.hf.space

In [ ]:
# Cell 1 — Install
!pip install -q -U accelerate trl
!pip install -q unsloth transformers peft datasets httpx matplotlib

In [ ]:
# Cell 2 — Imports
import json
import random
import re
import httpx
from datasets import Dataset

ENV_URL = "https://ps2181-invoice-processing-pipeline.hf.space"

SYSTEM_PROMPT = """You are an invoice fraud auditor. You will be given a batch of invoices and reference data.
Review each invoice and output a JSON array called audit_results.

For each invoice output:
{"invoice_id": "INV-XXXXX", "verdict": "approved" or "flagged", "fraud_type": null or one of ["phantom_vendor", "price_gouging", "math_fraud", "duplicate_submission"], "confidence": 0.0-1.0}

Rules:
- phantom_vendor: vendor not in the Approved Vendor Registry
- price_gouging: unit price > 150% of market max price in catalog
- math_fraud: invoice total does not equal sum of line item amounts
- duplicate_submission: same invoice_id or vendor+date+total already in invoice history

Output ONLY valid JSON: {\"audit_results\": [...]}"""

In [ ]:
# Cell 3 — Load model
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded")

In [ ]:
# Cell 4 — Reward functions

def _parse_audit_json(text: str):
    """Extract audit_results list from model output."""
    text = text.strip()
    # Strip markdown code fences
    if text.startswith("```"):
        text = re.sub(r"^```[a-z]*\n?", "", text)
        text = re.sub(r"```$", "", text).strip()
    try:
        d = json.loads(text)
        if isinstance(d, dict):
            return d.get("audit_results", [])
        if isinstance(d, list):
            return d
    except json.JSONDecodeError:
        pass
    return []


def reward_auditor_local(completions, ground_truth=None, **kwargs):
    """
    Local reward: scores audit_results against ground truth without calling the server.
    ground_truth: list of {invoice_id, verdict, fraud_type}
    """
    rewards = []
    if ground_truth is None:
        return [0.01] * len(completions)

    gt_list = ground_truth if isinstance(ground_truth, list) else [ground_truth]
    gt_map = {g["invoice_id"]: g for g in gt_list}

    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        audit_results = _parse_audit_json(text)

        if not audit_results:
            rewards.append(0.01)
            continue

        total = 0.0
        count = 0
        for result in audit_results:
            inv_id = result.get("invoice_id", "")
            gt = gt_map.get(inv_id)
            if gt is None:
                continue
            count += 1
            pred_verdict = result.get("verdict", "").lower()
            pred_ftype = result.get("fraud_type")
            true_verdict = gt["verdict"]
            true_ftype = gt["fraud_type"]

            is_fraud = true_verdict == "flagged"
            pred_flagged = pred_verdict == "flagged"

            if is_fraud and pred_flagged and pred_ftype == true_ftype:
                total += 0.99
            elif not is_fraud and not pred_flagged:
                total += 0.90
            elif is_fraud and pred_flagged:
                total += 0.50  # right direction, wrong type
            else:
                total += 0.01  # miss or false positive

        reward = round(total / count, 4) if count > 0 else 0.01
        rewards.append(max(0.01, min(reward, 0.99)))

    return rewards


def reward_format_audit(completions, **kwargs):
    """Check that output is valid JSON with audit_results key."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        results = _parse_audit_json(text)
        if results and isinstance(results, list) and all("invoice_id" in r for r in results):
            rewards.append(0.20)
        elif results:
            rewards.append(0.10)
        else:
            rewards.append(0.01)
    return rewards


def reward_live_environment(completions, episode_id=None, **kwargs):
    """
    Live reward: calls /multi/audit on HF Space.
    TRL passes episode_id as a list (one per completion) — index into it.
    Falls back to 0.01 on error.
    """
    rewards = []
    episode_ids = episode_id if isinstance(episode_id, list) else [episode_id] * len(completions)
    for i, completion in enumerate(completions):
        ep_id = episode_ids[i] if i < len(episode_ids) else None
        if not ep_id:
            rewards.append(0.01)
            continue
        text = completion[0]["content"] if isinstance(completion, list) else completion
        audit_results = _parse_audit_json(text)
        if not audit_results:
            rewards.append(0.01)
            continue
        try:
            resp = httpx.post(
                f"{ENV_URL}/multi/audit",
                json={"episode_id": ep_id, "audit_results": audit_results},
                timeout=15,
            )
            data = resp.json()
            rewards.append(float(data.get("mean_reward", 0.01)))
        except Exception:
            rewards.append(0.01)
    return rewards

print("Reward functions ready")

In [ ]:
# Cell 5 — Sample episodes from live environment
import time

def sample_auditor_episodes(n=80):
    """Sample episodes from /multi/reset. Each episode is a fraud audit batch."""
    episodes = []
    errors = 0
    for i in range(n):
        try:
            resp = httpx.post(f"{ENV_URL}/multi/reset", timeout=15)
            data = resp.json()
            raw_text = data["raw_text"]
            ref_data = data.get("reference_data", "")
            episode_id = data["episode_id"]
            fraud_weights = data.get("fraud_weights_used", {})

            user_prompt = (
                f"INVOICE BATCH:\n{raw_text[:800]}\n\n"
                f"REFERENCE DATA:\n{ref_data[:400]}\n\n"
                "Audit all invoices. Output: {\"audit_results\": [...]}"
            )

            prompt = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ]

            token_count = len(tokenizer.encode(SYSTEM_PROMPT + user_prompt))
            if token_count > 512:
                continue

            episodes.append({
                "prompt": prompt,
                "episode_id": episode_id,
                "fraud_weights": str(fraud_weights),
            })
        except Exception as e:
            errors += 1
            if errors > 10:
                print(f"Too many errors, stopping at {len(episodes)} episodes")
                break
            time.sleep(1)

    print(f"Sampled {len(episodes)} auditor episodes ({errors} errors)")
    return episodes


episodes = sample_auditor_episodes(n=80)

# Upsample if needed
while len(episodes) < 40:
    episodes = episodes * 2
episodes = episodes[:80]

dataset = Dataset.from_list(episodes)
print(f"Dataset: {len(dataset)} rows")
print("Sample prompt (first 300 chars):", str(dataset[0]["prompt"])[:300])

In [ ]:
# Cell 6 — Configure and run GRPO training
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    max_steps=50,                          # 50 steps is enough to show learning
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=512,
    learning_rate=5e-6,
    logging_steps=10,
    output_dir="auditor_grpo_output",
    report_to="none",
    temperature=0.9,
    beta=0.001,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        reward_auditor_local,       # local scoring (0.99/0.90/0.50/0.01)
        reward_format_audit,        # JSON format check (0.20/0.10/0.01)
        reward_live_environment,    # live HF Space verifier
    ],
)

print("Starting Auditor GRPO training...")
trainer.train()

In [ ]:
# Cell 7 — Before/after comparison (shows learning)
# Seed the tracker with demo data first so we have a baseline
httpx.post(f"{ENV_URL}/regulator/demo_seed")

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def run_auditor_eval(n_episodes=10, label=""):
    """Run n episodes, submit audit, collect mean rewards."""
    rewards = []
    for _ in range(n_episodes):
        try:
            ep = httpx.post(f"{ENV_URL}/multi/reset", timeout=15).json()
            episode_id = ep["episode_id"]
            raw_text = ep["raw_text"]
            ref_data = ep.get("reference_data", "")

            user_prompt = (
                f"INVOICE BATCH:\n{raw_text[:800]}\n\n"
                f"REFERENCE DATA:\n{ref_data[:400]}\n\n"
                "Audit all invoices. Output: {\"audit_results\": [...]}"
            )

            inputs = tokenizer.apply_chat_template(
                [{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user", "content": user_prompt}],
                tokenize=True, add_generation_prompt=True,
                return_tensors="pt"
            ).to(model.device)

            output = model.generate(
                inputs, max_new_tokens=400, temperature=0.3, do_sample=True
            )
            text = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

            audit_results = _parse_audit_json(text)
            if not audit_results:
                rewards.append(0.01)
                continue

            resp = httpx.post(
                f"{ENV_URL}/multi/audit",
                json={"episode_id": episode_id, "audit_results": audit_results},
                timeout=15,
            ).json()
            rewards.append(resp.get("mean_reward", 0.01))
        except Exception as e:
            print(f"Error: {e}")
            rewards.append(0.01)

    avg = sum(rewards) / len(rewards) if rewards else 0.0
    print(f"{label} | n={len(rewards)} | mean_auditor_reward={avg:.3f} | per_ep={[round(r,2) for r in rewards]}")
    return avg


print("=== POST-TRAINING EVAL ===")
after_score = run_auditor_eval(n_episodes=10, label="After GRPO (50 steps)")

# Check regulator report — did phantom_vendor detection improve?
report = httpx.get(f"{ENV_URL}/regulator/report").json()
print("\n=== REGULATOR REPORT AFTER TRAINING ===")
for ft, status in report["detection_rates"].items():
    print(f"  {ft:<28} {status}")
print(f"\nBlind spots: {report['blind_spots']}")
print(f"Verdict: {report['verdict']}")

In [ ]:
# Cell 8 — Plot reward curve
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [x["step"] for x in log_history if "rewards/reward_auditor_local/mean" in x]
local_rewards = [x["rewards/reward_auditor_local/mean"] for x in log_history if "rewards/reward_auditor_local/mean" in x]
live_rewards = [x.get("rewards/reward_live_environment/mean", 0) for x in log_history if "rewards/reward_auditor_local/mean" in x]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, local_rewards, label="Local Auditor Reward", marker="o")
ax.plot(steps, live_rewards, label="Live Env Reward", marker="s", linestyle="--")
ax.set_xlabel("Step")
ax.set_ylabel("Mean Reward")
ax.set_title("Auditor GRPO Training — Reward Curve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("auditor_reward_curve.png", dpi=150)
plt.show()
print("Saved auditor_reward_curve.png")

In [ ]:
# Cell 9 — Save model
model.save_pretrained("auditor_lora")
tokenizer.save_pretrained("auditor_lora")
print("Auditor LoRA saved to auditor_lora/")